# Clasificador OpenAI gpt-5-mini
Clasifica el dataset de preguntas de red usando la API de OpenAI.

In [12]:
pip install openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## Configuración

In [17]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../").resolve() / ".env")

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-5-mini"
INPUT_CSV = "requirements_questions_v2.csv"
OUTPUT_CSV = "requirements_classified_openai.csv"
CHECKPOINT_BASE = "checkpoint_openai.json"
SAMPLE = 20  # usa un int para prueba rápida

def build_run_checkpoint_path(base_path: str, model: str, input_path: str, sample: int | None) -> Path:
    base = Path(base_path)
    sample_tag = f"sample{sample}" if sample else "full"
    model_tag = re.sub(r"[^a-zA-Z0-9]+", "_", model).strip("_")
    input_tag = re.sub(r"[^a-zA-Z0-9]+", "_", Path(input_path).stem).strip("_")
    run_tag = f"{input_tag}_{model_tag}_{sample_tag}".lower()
    return base.with_name(f"{base.stem}_{run_tag}{base.suffix}")

CHECKPOINT = build_run_checkpoint_path(CHECKPOINT_BASE, MODEL, INPUT_CSV, SAMPLE)

print(f"API_KEY cargada: {'OK' if API_KEY else 'NO ENCONTRADA'}")
print(f"Checkpoint: {CHECKPOINT}")

API_KEY cargada: OK
Checkpoint: checkpoint_openai_requirements_questions_v2_gpt_5_mini_sample20.json


## Setup

In [18]:
import json
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

CAT_LIST = ["ROUTING", "SECURITY", "QOS", "CONNECTIVITY", "MONITORING", "GENERAL"]

SYSTEM_PROMPT = """You are a Cisco network expert and dataset annotator.
Classify the given network configuration question into exactly one of these categories:

ROUTING, SECURITY, QOS, CONNECTIVITY, MONITORING, GENERAL

Rules:
- Choose the SINGLE most relevant category based on the question PRIMARY intent.
- If the question spans two domains, pick the one the question is MAINLY asking about.
- Use GENERAL only when the question genuinely does not fit any specific technical domain.
- Respond with a JSON block only. No extra text after the JSON.

Response format:
```json
{\"category\": \"CATEGORY_NAME\"}
```"""

print("Setup OK")

Setup OK


## Funciones

In [ ]:
def parse_json_from_response(text: str) -> dict | None:
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"\{[^{}]*\"category\"[^{}]*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


def classify(question: str) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question},
            ],
            temperature=0.0,
            max_completion_tokens=128,
        )
        raw = response.choices[0].message.content or ""
        parsed = parse_json_from_response(raw)
        if parsed and parsed.get("category") in CAT_LIST:
            return parsed["category"]
        for cat in CAT_LIST:
            if cat in raw.upper():
                return cat
        print(f"\n[UNKNOWN] QUESTION:\n{question}\n[RAW]:\n{raw}\n")
        return "UNKNOWN"
    except Exception as e:
        print(f"\n[UNKNOWN] QUESTION:\n{question}\n[ERROR]: {e}\n")
        return "UNKNOWN"

print("Funciones OK")

Funciones OK


## Cargar dataset

In [20]:
df = pd.read_csv(INPUT_CSV)
if "id" not in df.columns:
    df.insert(0, "id", range(len(df)))

if SAMPLE:
    df = df.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
    print(f"Modo prueba: {SAMPLE} registros")

print(f"Total: {len(df)} preguntas")
df[["id", "question"]].head()

Modo prueba: 20 registros
Total: 20 preguntas


,id,question
0,1552,What type of authentication do we need to impl...
1,23,How can I configure the percentile option for ...
2,252,What is the primary purpose of configuring NAT...
3,2698,What is the TCP connection timeout period set ...
4,3995,How can we monitor and classify traffic by the...


## Clasificar

In [ ]:
checkpoint = Path(CHECKPOINT)
results: dict[int, str] = {}
if checkpoint.exists():
    with open(checkpoint) as f:
        results = {int(k): v for k, v in json.load(f).items()}
    print(f"Checkpoint: {len(results)} registros ya clasificados.")

all_ids = [int(x) for x in df["id"].tolist()]
# Reintenta registros faltantes o con UNKNOWN previo.
pending = [row_id for row_id in all_ids if row_id not in results or results.get(row_id) == "UNKNOWN"]
print(f"Pendientes (incluye UNKNOWN previos): {len(pending)}")

for count, row_id in enumerate(tqdm(pending, total=len(pending))):
    row = df[df["id"] == row_id].iloc[0]
    question = str(row["question"])
    results[row_id] = classify(question)

    if (count + 1) % 50 == 0:
        with open(checkpoint, "w") as f:
            json.dump(results, f)
        unknown = sum(1 for c in results.values() if c == "UNKNOWN")
        print(f"[{len(results)}/{len(all_ids)}] guardado | UNKNOWN: {unknown}")

with open(checkpoint, "w") as f:
    json.dump(results, f)

print("Clasificación completa")

Pendientes: 20


  0%|          | 0/20 [00:00<?, ?it/s]


[UNKNOWN] QUESTION:
What type of authentication do we need to implement on neighbor routers in routing updates, considering security concerns?
[ERROR]: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}


[UNKNOWN] QUESTION:
How can I configure the percentile option for IP SLAs operations?
[ERROR]: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}


[UNKNOWN] QUESTION:
What is the primary purpose of configuring NAT in an IP network?
[ERROR]: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request

## Exportar y revisar resultados

In [11]:
df["category"] = [results.get(int(row_id), "UNKNOWN") for row_id in df["id"]]
df.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset guardado: {OUTPUT_CSV}")

unknown_count = int((df["category"] == "UNKNOWN").sum())
print(f"UNKNOWN: {unknown_count}")

print("\n=== DISTRIBUCIÓN FINAL ===")
display(df["category"].value_counts().to_frame())

df[["id", "question", "category"]].head(10)

Dataset guardado: requirements_classified_openai.csv
UNKNOWN: 0

=== DISTRIBUCIÓN FINAL ===


,count
category,
SECURITY,5
MONITORING,5
QOS,5
ROUTING,3
GENERAL,1
CONNECTIVITY,1


,id,question,category
0,1552,What type of authentication do we need to impl...,SECURITY
1,23,How can I configure the percentile option for ...,MONITORING
2,252,What is the primary purpose of configuring NAT...,GENERAL
3,2698,What is the TCP connection timeout period set ...,MONITORING
4,3995,How can we monitor and classify traffic by the...,MONITORING
5,3547,What is the command sequence to replace MQC Co...,QOS
6,1117,What steps can you take to configure VXLAN HA ...,CONNECTIVITY
7,1366,What is the IP address of the management stati...,MONITORING
8,2699,How can I display VRF-level policy firewall st...,MONITORING
9,1860,How do you configure PfRv3 on the hub-border r...,QOS
